# Data Science 1 Final Project

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

df = pd.read_csv("dataset.csv")

## Data Cleaning

In [2]:
broad_genre_map = {
    
    # Pop-ish music
    "pop": "pop",
    "power-pop": "pop",
    "synth-pop": "pop",
    "indie-pop": "pop",
    "k-pop": "pop",
    "j-pop": "pop",
    "pop-film": "pop",
    "children": "pop",
    "kids": "pop",
    "j-dance": "pop",
    "j-idol": "pop",
    "mandopop": "pop",
    "cantopop": "pop",

    # Rock and metal
    "rock": "rock",
    "alt-rock": "rock",
    "alternative": "other", #rock was my worst performing on everything so i went back and killed the more genre ambiguous subgenres
    "indie": "other",       #same as above
    "punk": "rock",
    "punk-rock": "rock",
    "psych-rock": "rock",
    "hard-rock": "rock",
    "rock-n-roll": "rock",
    "rockabilly": "rock",
    "grunge": "rock",
    "metal": "rock",
    "heavy-metal": "rock",
    "black-metal": "rock",
    "death-metal": "rock",
    "metalcore": "rock",
    "goth": "rock",
    "emo": "rock",
    "grindcore": "rock",

    # Hip-hop / R&B
    "hip-hop": "hiphop/rnb",
    "r-n-b": "hiphop/rnb",
    "soul": "hiphop/rnb",
    "funk": "hiphop/rnb",
    "groove": "other",
    

    # Electronic
    "edm": "electronic",
    "electronic": "electronic",
    "electro": "electronic",
    "house": "electronic",
    "deep-house": "electronic",
    "progressive-house": "electronic",
    "chicago-house": "electronic",
    "minimal-techno": "electronic",
    "techno": "electronic",
    "trance": "electronic",
    "dubstep": "electronic",
    "drum-and-bass": "electronic",
    "breakbeat": "electronic",
    "idm": "electronic",
    "club": "electronic",
    "dance": "electronic",
    "dancehall": "electronic",
    "garage": "electronic",
    "trip-hop": "electronic",

    # Latin
    "latin": "latin",
    "latino": "latin",
    "salsa": "latin",
    "samba": "latin",
    "pagode": "latin",
    "sertanejo": "latin",
    "mpb": "latin",
    "brazil": "latin",
    "forro": "latin",
    "reggaeton": "latin",
    "reggae": "latin",
    "tango": "latin",

    # Jazz/ Blues
    "jazz": "jazz-blues",
    "blues": "jazz-blues",
    
    # Country
    "bluegrass": "country",
    "folk": "country",
    "country": "country",
    "honky-tonk": "country",

    # Classical
    "classical": "classical",
    "opera": "classical",
    "piano": "classical",
    "ambient": "classical",
    "new-age": "classical",

    # other
    "afrobeat": "other",
    "indian": "other",
    "iranian": "other",
    "turkish": "other",
    "malay": "other",
    "world-music": "other",
    "french": "other",
    "german": "other",
    "spanish": "other",
    "swedish": "other",
    "acoustic": "other",
    "chill": "other",
    "happy": "other",
    "sad": "other",
    "sleep": "other",
    "study": "other",
    "party": "other",
    "romance": "other",
    "gospel": "other",
    "comedy": "other",
    "disco": "other",
    "disney": "other",
    "show-tunes": "other",
    "guitar": "other",
    "songwriter": "other",
    "singer-songwriter": "other"
}

df["broad_genre"] = df["track_genre"].map(broad_genre_map)
   
major_genres = ["pop", "rock", "hiphop/rnb", "electronic", "latin"]
df = df[df["broad_genre"].isin(major_genres)]

print(df["broad_genre"].value_counts())
print("data left after genre filtering:", len(df))


targetColumn = "popularity"
genreColumn = "track_genre" #this will change soon
    #already ran this so it wont show anything in the output
df.drop(columns=["artists", "track_id", "album_name", "track_name"], inplace=True, errors="ignore")
df.drop(columns=["Unnamed: 0"], inplace=True, errors="ignore")
    # df.drop(columns=["failure", "Failure"])
    



featureColumns = [
    col for col in df.columns if col not in [targetColumn, "failure", "broad_genre"]
]

print("Features:", featureColumns)

failureThreshold = 30 # defines failure as a popularity score below 20 (something like 30% the data. Also not a 50/50 split make bayes more interesting)
df["failure"] = (df[targetColumn] < failureThreshold).astype(int)

df[["popularity", "failure"]].head()
df["failure"].value_counts(normalize=True)

# get rid of specific genres and group them into major genres 
# and also kill the weird ones that are hard to classify.
# This is manual because i would need to do a whole other MLA to figure it out automatically




broad_genre
electronic    19000
rock          17000
pop           13000
latin         12000
hiphop/rnb     4000
Name: count, dtype: int64
data left after genre filtering: 65000
Features: ['duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']


failure
0    0.568369
1    0.431631
Name: proportion, dtype: float64

## Random Forest

In [3]:
X = df.drop(columns=["popularity", "failure", "track_genre", "broad_genre"])
y = df["broad_genre"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

pred = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print("Macro F1:", round(f1_score(y_test, pred, average="macro"), 3))

Accuracy: 0.705
Macro F1: 0.643
